In [1]:
import logging
import random
from collections import Counter, defaultdict
import numpy as np
import math
import random
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import ClassLabel, Dataset, DatasetDict, concatenate_datasets, load_dataset

logging.basicConfig(level=logging.INFO, format="%(message)s")
sns.set_style("whitegrid")

PATH = "codeparrot/apps"
EMBEDDING_PATH = 'Qwen/Qwen3-Embedding-0.6B'

In [2]:
dataset = load_dataset(PATH)

Using the latest cached version of the dataset since codeparrot/apps couldn't be found on the Hugging Face Hub
Using the latest cached version of the dataset since codeparrot/apps couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at /Users/joanvelja/.cache/huggingface/datasets/codeparrot___apps/all/0.0.0/04ac807715d07d6e5cc580f59cdc8213cd7dc4529d0bb819cca72c9f8e8c1aa5 (last modified on Mon Aug  4 18:34:53 2025).
Found the latest cached dataset configuration 'all' at /Users/joanvelja/.cache/huggingface/datasets/codeparrot___apps/all/0.0.0/04ac807715d07d6e5cc580f59cdc8213cd7dc4529d0bb819cca72c9f8e8c1aa5 (last modified on Mon Aug  4 18:34:53 2025).


In [3]:
combined = concatenate_datasets([dataset["train"], dataset["test"]])

In [4]:
unique_difficulties = list(set(dataset["train"]["difficulty"]))

counts = {difficulty: 0 for difficulty in unique_difficulties}
counts_test = {difficulty: 0 for difficulty in unique_difficulties}
counts_train = {difficulty: 0 for difficulty in unique_difficulties}

for difficulty in unique_difficulties:
    print(
        f"Number of training examples with difficulty {difficulty}: {dataset['train']['difficulty'].count(difficulty)}"
    )
    print(f"Number of test examples with difficulty {difficulty}: {dataset['test']['difficulty'].count(difficulty)}")
    counts[difficulty] = dataset["train"]["difficulty"].count(difficulty) + dataset["test"]["difficulty"].count(
        difficulty
    )
    counts_train[difficulty] = dataset["train"]["difficulty"].count(difficulty)
    counts_test[difficulty] = dataset["test"]["difficulty"].count(difficulty)

Number of training examples with difficulty competition: 361
Number of test examples with difficulty competition: 1000
Number of training examples with difficulty introductory: 2639
Number of test examples with difficulty introductory: 1000
Number of training examples with difficulty interview: 2000
Number of test examples with difficulty interview: 3000


In [5]:
counts

{'competition': 1361, 'introductory': 3639, 'interview': 5000}

In [6]:
# ---------- helpers (unchanged from previous version) ----------
def _dist(c):
    t = sum(c.values()) or 1
    return {k: v / t for k, v in c.items()}


def _l1(p, q, keys=None):
    if keys is None:
        keys = set(p) | set(q)
    return sum(abs(p.get(k, 0) - q.get(k, 0)) for k in keys)


def _project_capped_simplex(m, u, s, tol=1e-9, iters=80):
    m = np.asarray(m, float)
    u = np.asarray(u, float)
    s = float(min(max(s, 0.0), u.sum()))
    lo, hi = np.min(-m), np.max(u - m)
    f = lambda lam: np.clip(m + lam, 0, u).sum()
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        val = f(mid)
        if abs(val - s) <= tol:
            break
        hi, lo = (mid, lo) if val > s else (hi, mid)
    return np.clip(m + 0.5 * (lo + hi), 0, u)


def _integerize(x, u, target):
    x = np.floor(x).astype(int)
    rem = x - x  # placeholder for vectorised view
    rem = x.astype(float)
    rem -= np.floor(rem)  # remaining frac – cheap, keeps shape
    deficit = int(round(target - x.sum()))
    idx_inc = np.argsort(-rem)
    idx_dec = np.argsort(rem)
    while deficit > 0:
        for i in idx_inc:
            if deficit == 0:
                break
            if x[i] < u[i]:
                x[i] += 1
                deficit -= 1
    while deficit < 0:
        for i in idx_dec:
            if deficit == 0:
                break
            if x[i] > 0:
                x[i] -= 1
                deficit += 1
    assert x.sum() == target, "integerisation failed"
    return x


def _fit_x(Ng, Ttr, p_tr, p_te, alpha):
    """Closed-form, then projection."""
    Tte = float(Ng.sum() - Ttr)
    Ng = Ng.astype(float)
    m = alpha * Ttr * p_tr + (1 - alpha) * (Ng - Tte * p_te)
    x = _project_capped_simplex(m / (alpha + (1 - alpha)), Ng, Ttr)
    return x


def _eval_loss(x, Ng, Ttr, keys, tr_dist, te_dist):
    y = Ng - x
    Tte = int(Ng.sum() - Ttr)
    tr_new = {k: x[i] / Ttr for i, k in enumerate(keys)}
    te_new = {k: y[i] / Tte for i, k in enumerate(keys)}
    l1_tr = _l1(tr_dist, tr_new, keys)
    l1_te = _l1(te_dist, te_new, keys)
    return 0.5 * (l1_tr + l1_te), l1_tr, l1_te, tr_new, te_new


# ---------- golden-section on α ----------
def _best_alpha(Ng, Ttr, p_tr, p_te, keys, tr_dist, te_dist, tol=1e-3):
    φ = (math.sqrt(5) - 1) / 2
    a, b = 0.0, 1.0
    f_cache = {}

    def f(alpha):
        if alpha not in f_cache:
            x = _fit_x(Ng, Ttr, p_tr, p_te, alpha)
            loss, _1, _2, _3, _4 = _eval_loss(x, Ng, Ttr, keys, tr_dist, te_dist)
            f_cache[alpha] = loss
        return f_cache[alpha]

    x1 = b - φ * (b - a)
    x2 = a + φ * (b - a)
    f1, f2 = f(x1), f(x2)
    while b - a > tol:
        if f1 > f2:
            a, x1, f1 = x1, x2, f2
            x2 = a + φ * (b - a)
            f2 = f(x2)
        else:
            b, x2, f2 = x2, x1, f1
            x1 = b - φ * (b - a)
            f1 = f(x1)
    α = (a + b) / 2
    loss = f(α)
    return α, loss


# ---------- main ----------
def reshuffle_stratified(
    ds: DatasetDict, feature="difficulty", target_train_frac=0.8, relax_frac=0.02, alpha_tol=1e-3, seed=0
):
    assert {"train", "test"} <= set(ds), "need 'train' and 'test' splits"
    rng = np.random.default_rng(seed)
    tr_vals, te_vals = ds["train"][feature], ds["test"][feature]
    tr_dist, te_dist = _dist(Counter(tr_vals)), _dist(Counter(te_vals))

    pool = concatenate_datasets([ds["train"], ds["test"]])
    values = pool[feature]
    buckets = defaultdict(list)
    for i, v in enumerate(values):
        buckets[v].append(i)
    keys = sorted(buckets)
    Ng = np.array([len(buckets[k]) for k in keys])
    N = int(Ng.sum())
    p_tr = np.array([tr_dist.get(k, 0.0) for k in keys])
    p_te = np.array([te_dist.get(k, 0.0) for k in keys])

    T0 = int(round(target_train_frac * N))
    window = int(relax_frac * N)
    cand_T = range(max(0, T0 - window), min(N, T0 + window) + 1)

    best = dict(loss=float("inf"))
    for Ttr in cand_T:
        α, loss = _best_alpha(Ng, Ttr, p_tr, p_te, keys, tr_dist, te_dist, tol=alpha_tol)
        x_float = _fit_x(Ng, Ttr, p_tr, p_te, α)
        x = _integerize(x_float, Ng, Ttr)
        final_loss, l1tr, l1te, tr_new, te_new = _eval_loss(x, Ng, Ttr, keys, tr_dist, te_dist)
        closer = abs(Ttr - T0) < abs(best.get("Ttr", T0) - T0)
        if final_loss < best["loss"] - 1e-12 or (abs(final_loss - best["loss"]) < 1e-12 and closer):
            best.update(dict(Ttr=Ttr, α=α, x=x, loss=final_loss, l1tr=l1tr, l1te=l1te, tr_new=tr_new, te_new=te_new))

    # materialise indices
    train_idx, test_idx = [], []
    for k, take in zip(keys, best["x"]):
        bucket = buckets[k][:]
        rng.shuffle(bucket)
        train_idx.extend(bucket[: int(take)])
        test_idx.extend(bucket[int(take) :])
    train_idx.sort()
    test_idx.sort()
    out = DatasetDict(train=pool.select(train_idx), test=pool.select(test_idx))

    # ---- prints ----
    print(f"Total N={N}. Requested {100*target_train_frac:.2f}% train, radius ±{100*relax_frac:.1f}%")
    print(
        f"Best split: train={best['Ttr']} ({100*best['Ttr']/N:.2f}%), " f"test={N-best['Ttr']}  |  α*={best['α']:.3f}"
    )
    print(f"  L1 divergence -> train={best['l1tr']:.4f}, test={best['l1te']:.4f}, avg={best['loss']:.4f}")
    top = lambda d: ", ".join(f"{k}:{v:.3f}" for k, v in sorted(d.items(), key=lambda kv: -kv[1])[:5])
    print("Original train dist (top):", top(tr_dist))
    print("New      train dist (top):", top(best["tr_new"]))
    print("Original test  dist (top):", top(te_dist))
    print("New      test  dist (top):", top(best["te_new"]))
    return out

In [7]:
new_ds = reshuffle_stratified(dataset, feature="difficulty", target_train_frac=0.9, relax_frac=0.05, seed=42)

Total N=10000. Requested 90.00% train, radius ±5.0%
Best split: train=8504 (85.04%), test=1496  |  α*=0.000
  L1 divergence -> train=0.2701, test=0.0005, avg=0.1353
Original train dist (top): introductory:0.528, interview:0.400, competition:0.072
New      train dist (top): interview:0.482, introductory:0.393, competition:0.125
Original test  dist (top): interview:0.600, competition:0.200, introductory:0.200
New      test  dist (top): interview:0.600, competition:0.200, introductory:0.200


In [8]:
# new_ds.push_to_hub("jvelja/apps_reshuffled")

In [9]:
import ast


def parse_code_snippets(raw: str) -> list[str]:
    """
    Turn a string like the example into a list of code-snippet strings.
    Removes lone surrogate code points to avoid UnicodeEncodeError on encoding.
    """

    def remove_surrogates(s: str) -> str:
        # drop any isolated surrogate code points in U+D800..U+DFFF
        return "".join(ch for ch in s if not (0xD800 <= ord(ch) <= 0xDFFF))

    # Primary path: try to interpret the whole thing as a Python list literal.
    try:
        lst = ast.literal_eval(raw)
        if isinstance(lst, str) or not isinstance(lst, list):
            raise ValueError("Not a list of strings")
        cleaned = []
        for snippet in lst:
            if not isinstance(snippet, str):
                snippet = str(snippet)
            cleaned.append(remove_surrogates(snippet))
        return cleaned
    except Exception:
        # Fallback: manually extract top-level string literals (handles malformed cases).
        snippets = []
        i = 0
        L = len(raw)
        while i < L:
            if raw[i] in ('"', "'"):
                quote = raw[i]
                i += 1
                buf = []
                escaped = False
                while i < L:
                    c = raw[i]
                    if escaped:
                        buf.append(c)
                        escaped = False
                    elif c == "\\":
                        buf.append(c)
                        escaped = True
                    elif c == quote:
                        # closing quote
                        i += 1
                        literal = quote + "".join(buf) + quote
                        try:
                            val = ast.literal_eval(literal)
                        except Exception:
                            val = "".join(buf)
                        if isinstance(val, str):
                            val = remove_surrogates(val)
                        else:
                            val = remove_surrogates(str(val))
                        snippets.append(val)
                        break
                    else:
                        buf.append(c)
                    i += 1
            else:
                i += 1
        if not snippets:
            # ultimate fallback: return the raw string cleaned
            return [remove_surrogates(raw)]
        return snippets


def sanitize_example(example):
    # Assumes the raw column is named 'solutions'; adjust if different.
    raw = example.get("solutions", "")
    example["solutions_list"] = parse_code_snippets(raw)
    return example

In [10]:
new_sanitized_ds = new_ds.map(sanitize_example)

In [11]:
new_sanitized_ds

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list'],
        num_rows: 8504
    })
    test: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list'],
        num_rows: 1496
    })
})

In [12]:
new_sanitized_ds["train"]["solutions_list"][0]

['q=int(input())\n\nfor e in range(q):\n    x,y,k=list(map(int,input().split()))\n    x,y=abs(x),abs(y)\n    x,y=max(x,y),min(x,y)\n    \n    if(x%2!=k%2):\n        k-=1\n        y-=1\n    \n    \n    if(x>k):\n        print(-1)\n        continue\n    if((x-y)%2):\n        k-=1\n        x-=1\n    print(k)\n    \n    \n    \n',
 "#      \nimport collections, atexit, math, sys, bisect \n\nsys.setrecursionlimit(1000000)\ndef getIntList():\n    return list(map(int, input().split()))    \n\ntry :\n    #raise ModuleNotFoundError\n    import numpy\n    def dprint(*args, **kwargs):\n        print(*args, **kwargs, file=sys.stderr)\n    dprint('debug mode')\nexcept ModuleNotFoundError:\n    def dprint(*args, **kwargs):\n        pass\n\n\n\ninId = 0\noutId = 0\nif inId>0:\n    dprint('use input', inId)\n    sys.stdin = open('input'+ str(inId) + '.txt', 'r') #标准输出重定向至文件\nif outId>0:\n    dprint('use output', outId)\n    sys.stdout = open('stdout'+ str(outId) + '.txt', 'w') #标准输出重定向至文件\n    atexit.

In [13]:
new_sanitized_ds

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list'],
        num_rows: 8504
    })
    test: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list'],
        num_rows: 1496
    })
})

In [14]:
new_sanitized_ds = new_sanitized_ds.map(lambda ex: {"n_sols": len(ex["solutions_list"])}, batched=False)

In [15]:
# Extract the counts and print summary
counts = np.array(new_sanitized_ds["train"]["n_sols"])
print("Total problems:", len(counts))
print("Min solutions:", counts.min())
print("Max solutions:", counts.max())
print("Median:", np.median(counts))
print("Mean:", counts.mean())
print("Stddev:", counts.std())

Total problems: 8504
Min solutions: 1
Max solutions: 990
Median: 10.0
Mean: 23.892756349952965
Stddev: 39.174685058223524


In [16]:
# Extract the counts and print summary
test_counts = np.array(new_sanitized_ds["test"]["n_sols"])
print("Total problems:", len(test_counts))
print("Min solutions:", test_counts.min())
print("Max solutions:", test_counts.max())
print("Median:", np.median(test_counts))
print("Mean:", test_counts.mean())
print("Stddev:", test_counts.std())

Total problems: 1496
Min solutions: 1
Max solutions: 453
Median: 10.0
Mean: 20.384358288770052
Stddev: 31.39403697322005


In [17]:
print(new_sanitized_ds["train"]["solutions_list"][0][0].strip())

q=int(input())

for e in range(q):
    x,y,k=list(map(int,input().split()))
    x,y=abs(x),abs(y)
    x,y=max(x,y),min(x,y)
    
    if(x%2!=k%2):
        k-=1
        y-=1
    
    
    if(x>k):
        print(-1)
        continue
    if((x-y)%2):
        k-=1
        x-=1
    print(k)


In [18]:
print(new_sanitized_ds["train"]["solutions_list"][1][4].strip())

gans = []
for _ in range(int(input())):
    n = int(input())
    a = list(map(int, input().split()))
    b = list(map(int, input().split()))
    c = list(map(int, input().split()))
    ans = [a[0]]
    for i in range(1, n - 1):
        if a[i] != ans[i - 1]:
            ans.append(a[i])
        else:
            ans.append(b[i])
    if a[-1] != ans[-1] and a[-1] != ans[0]:
        ans.append(a[-1])
    elif b[-1] != ans[-1] and b[-1] != ans[0]:
        ans.append(b[-1])
    else:
        ans.append(c[-1])
    gans.append(' '.join(map(str, ans)))
print('\n'.join(gans))


In [19]:
import ast
from collections import ChainMap

# --- constant sets (unchanged) ------------------------------------------
BUILTIN_FUNCS = {"input", "raw_input"}
STDIN_READ_ATTRS = {"read", "readline", "readlines", "readinto", "readall"}
INTERACTIVE_MODULES = {
    "curses",
    "keyboard",
    "prompt_toolkit",
    "tkinter",
    "pygame",
    "pynput",
    "msvcrt",
    "termios",
    "tty",
    "click",
    "inquirer",
    "questionary",
}
SUBPROCESS_FUNCS = {
    ("subprocess", "run"),
    ("subprocess", "call"),
    ("subprocess", "Popen"),
    ("subprocess", "check_output"),
    ("subprocess", "check_call"),
}
SUBPROC_BARE_NAMES = {"run", "call", "Popen", "check_output", "check_call"}
LIKELY_TTY_CMDS = {"read", "less", "more", "vi", "vim", "nano", "bash", "sh", "python"}
EXTERNAL_INTERPRETERS = {
    "python",
    "python3",
    "pypy",
    "pypy3",
    "ruby",
    "irb",
    "julia",
    "node",
    "java",
    "lua",
    "perl",
    "php",
    "racket",
    "ghci",
}
LOW_LEVEL_STDIN_FUNCS = {("os", "read"), ("os", "fdopen")}
TTY_DEVICE_PATHS = {"/dev/tty", "/dev/stdin", "CON", "CON:"}
FILEINPUT_CALL = ("fileinput", "input")


# --- tiny helpers (unchanged) -------------------------------------------
def _attr_chain(node):
    parts = []
    while isinstance(node, ast.Attribute):
        parts.append(node.attr)
        node = node.value
    if isinstance(node, ast.Name):
        parts.append(node.id)
    return list(reversed(parts))


# --- InteractionFinder class (UPDATED) --------------------------------
class InteractionFinder(ast.NodeVisitor):
    def __init__(self):
        self.found, self.possible = False, []
        self.scopes = [ChainMap()]
        self.mod_alias = [{}]
        self.wild_sys = [False]
        self.wild_subp = [False]
        super().__init__()

    def _push(self):
        self.scopes.append(self.scopes[-1].new_child())
        self.mod_alias.append(self.mod_alias[-1].copy())
        self.wild_sys.append(self.wild_sys[-1])
        self.wild_subp.append(self.wild_subp[-1])

    def _pop(self):
        self.scopes.pop()
        self.mod_alias.pop()
        self.wild_sys.pop()
        self.wild_subp.pop()

    def _mark(self, node, msg):
        if not self.found:
            self.found = True
        self.possible.append((node.lineno, node.col_offset, msg))

    def _lookup(self, name):
        for scope in reversed(self.scopes):
            if name in scope:
                return scope[name]
        if name == "stdin" and any(self.wild_sys):
            return "stdin"
        if name in SUBPROC_BARE_NAMES and any(self.wild_subp):
            return "subproc"
        if name in BUILTIN_FUNCS:
            return "builtin_input"
        return None

    def _get_import_chain(self, node):
        if not isinstance(node, ast.Attribute):
            return None
        chain = []
        curr = node
        while isinstance(curr, ast.Attribute):
            chain.append(curr.attr)
            curr = curr.value
        if (
            isinstance(curr, ast.Call)
            and isinstance(curr.func, ast.Name)
            and curr.func.id == "__import__"
            and curr.args
            and isinstance(curr.args[0], ast.Constant)
            and isinstance(curr.args[0].value, str)
        ):
            mod_name = curr.args[0].value
            chain.append(mod_name)
            chain.reverse()
            return chain
        return None

    def _is_stdin_expr(self, node):
        if isinstance(node, ast.Name):
            return self._lookup(node.id) == "stdin"
        if isinstance(node, ast.Attribute):
            ch = _attr_chain(node)
            if not ch:
                return False
            base, real = ch[0], self.mod_alias[-1].get(ch[0], ch[0])
            return (real == "sys" and ch[1] == "stdin") or self._lookup(base) == "stdin"
        return False

    def _is_stdin_fd_expr(self, node):
        if isinstance(node, ast.Constant) and node.value == 0:
            return True
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "fileno":
            return self._is_stdin_expr(node.func.value)
        return False

    def visit_Import(self, node):
        for n in node.names:
            self.mod_alias[-1][n.asname or n.name.split(".")[0]] = n.name.split(".")[0]
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        mod = (node.module or "").split(".")[0]
        if len(node.names) == 1 and node.names[0].name == "*":
            if mod == "sys":
                self.wild_sys[-1] = True
            if mod == "subprocess":
                self.wild_subp[-1] = True
            self.generic_visit(node)
            return
        for n in node.names:
            local = n.asname or n.name
            if mod == "sys" and n.name == "stdin":
                self.scopes[-1][local] = "stdin"
            elif mod in INTERACTIVE_MODULES:
                self.scopes[-1][local] = "module"
            elif mod == "click" and n.name in {"prompt", "confirm"}:
                self.scopes[-1][local] = "callable"
            elif mod == "os" and n.name in {"read", "fdopen"}:
                self.scopes[-1][local] = f"os_{n.name}"
            elif mod == "fileinput" and n.name == "input":
                self.scopes[-1][local] = "fileinput"
            elif mod == "subprocess" and n.name in SUBPROC_BARE_NAMES:
                self.scopes[-1][local] = "subproc"
        self.generic_visit(node)

    def _alias(self, target, value):
        if isinstance(target, (ast.Tuple, ast.List)):
            if isinstance(value, (ast.Tuple, ast.List)) and len(target.elts) == len(value.elts):
                for t, v in zip(target.elts, value.elts):
                    self._alias(t, v)
            return
        if not isinstance(target, ast.Name):
            return
        if isinstance(value, ast.Name):
            k = self._lookup(value.id)
            if k:
                self.scopes[-1][target.id] = k
                return
        if isinstance(value, ast.Attribute):
            import_chain = self._get_import_chain(value)
            if import_chain:
                base, parts = import_chain[0], import_chain[1:]
                if base == "sys" and parts and parts[0] == "stdin":
                    if len(parts) > 1 and parts[1] in STDIN_READ_ATTRS:
                        self.scopes[-1][target.id] = "callable"
                    elif len(parts) == 1:
                        self.scopes[-1][target.id] = "stdin"
                    return
            ch = _attr_chain(value)
            if not ch:
                return
            base = ch[0]
            real = self.mod_alias[-1].get(base, base)
            if real == "sys" and ch[1] == "stdin":
                self.scopes[-1][target.id] = "stdin" if ch[-1] not in STDIN_READ_ATTRS else "callable"
                return
            if real == "click" and ch[-1] in {"prompt", "confirm"}:
                self.scopes[-1][target.id] = "callable"
                return
            if real in INTERACTIVE_MODULES:
                self.scopes[-1][target.id] = "module"
                return

    def visit_Assign(self, n):
        for t in n.targets:
            self._alias(t, n.value)
        self.generic_visit(n)

    def visit_AnnAssign(self, n):
        self._alias(n.target, n.value)
        self.generic_visit(n)

    def visit_Name(self, n):
        k = self._lookup(n.id)
        if k == "stdin":
            self._mark(n, "stdin alias")
        elif k == "builtin_input":
            self._mark(n, "builtin input()")
        self.generic_visit(n)

    def visit_Attribute(self, n):
        if self._is_stdin_expr(n):
            self._mark(n, "sys.stdin")
        self.generic_visit(n)

    def _tty_cmd(self, call):
        for kw in call.keywords:
            if kw.arg == "stdin":
                return False
        if not call.args:
            return False
        first = call.args[0]
        if (
            isinstance(first, (ast.List, ast.Tuple))
            and first.elts
            and isinstance(first.elts[0], ast.Constant)
            and isinstance(first.elts[0].value, str)
        ):
            cmd = first.elts[0].value
        elif isinstance(first, ast.Constant) and isinstance(first.value, str):
            cmd = first.value.split()[0]
        else:
            cmd = None
        return cmd and (cmd in LIKELY_TTY_CMDS or cmd in EXTERNAL_INTERPRETERS)

    def visit_Call(self, n):
        if self.found:
            return
        f = n.func
        if isinstance(f, ast.Name):
            k = self._lookup(f.id)
            if k in {"callable", "fileinput", "subproc", "os_read", "os_fdopen"} or k == "builtin_input":
                if k == "subproc" and not self._tty_cmd(n):
                    pass
                else:
                    self._mark(n, f"{f.id}()")
                    return

            if f.id == "open" and n.args:
                arg_node = n.args[0]
                # Handle walrus operator `(c:=0)` by looking at its value
                if isinstance(arg_node, ast.NamedExpr):
                    arg_node = arg_node.value

                if isinstance(arg_node, ast.Constant):
                    if isinstance(arg_node.value, int) and arg_node.value == 0:
                        self._mark(n, "open(stdin fd)")
                        return
                    if isinstance(arg_node.value, str) and arg_node.value in TTY_DEVICE_PATHS:
                        self._mark(n, "open('/dev/tty')")
                        return
        if isinstance(f, ast.Attribute):
            import_chain = self._get_import_chain(f)
            if import_chain:
                base, method = import_chain[0], import_chain[-1]
                if (base, method) in {("getpass", "getpass")}:
                    self._mark(n, f"{base}.{method}()")
                    return
                if base == "sys" and "stdin" in import_chain and method in STDIN_READ_ATTRS:
                    self._mark(n, "stdin via __import__")
                    return
            ch = _attr_chain(f)
            if not ch:
                self.generic_visit(n)
                return
            base, real = ch[0], self.mod_alias[-1].get(ch[0], ch[0])
            if real == "sys" and ch[1] == "stdin" and ch[-1] in STDIN_READ_ATTRS:
                self._mark(n, "sys.stdin.read*()")
                return
            if self._lookup(base) == "stdin" and ch[-1] in STDIN_READ_ATTRS:
                self._mark(n, "stdin-alias.read*()")
                return
            if (real, ch[-1]) in SUBPROCESS_FUNCS and self._tty_cmd(n):
                self._mark(n, "subprocess interactive")
                return
            if (real, ch[-1]) in LOW_LEVEL_STDIN_FUNCS and n.args and self._is_stdin_fd_expr(n.args[0]):
                self._mark(n, f"{real}.{ch[-1]}(stdin fd)")
                return
            if (real, ch[-1]) == FILEINPUT_CALL:
                self._mark(n, "fileinput.input()")
                return
            if real in INTERACTIVE_MODULES:
                self._mark(n, f"call into {real}")
                return
            if (real, ch[-1]) in {("getpass", "getpass"), ("click", "prompt"), ("click", "confirm")}:
                self._mark(n, f"{real}.{ch[-1]}()")
                return
        self.generic_visit(n)

    visit_FunctionDef = visit_AsyncFunctionDef = visit_Lambda = visit_ClassDef = lambda self, n: (
        self._push(),
        self.generic_visit(n),
        self._pop(),
    )

In [20]:
def analyze_interaction_strict(code: str) -> tuple[bool | None, str | None]:
    """
    Parses code and returns its interaction status.

    Returns:
        - (True, "reason"): If the code is interactive.
        - (False, None): If the code is provably non-interactive.
        - (None, "error_reason"): If the code is empty or has a syntax error.
    """
    if not isinstance(code, str) or not code.strip():
        return None, "not-a-str-or-empty"
    try:
        tree = ast.parse(code, feature_version=(3, 11))
    except SyntaxError as e:
        return None, f"syntax-error: {e.msg}"

    finder = InteractionFinder()  # Assuming InteractionFinder is defined as before
    finder.visit(tree)

    if finder.found:
        return True, finder.possible[0][2]
    else:
        return False, None

In [21]:
def detect_requires_input(example: dict) -> dict:
    """
    For `example["solutions_list"]`, determines non-interactive indices.

    This function operates with a strict, context-aware logic:
    1. It first analyzes the `starter_code`.
    2. If the starter code is interactive, all solutions are considered
       interactive by context, and their individual code is not analyzed.
    3. If the starter code is non-interactive or has errors, each solution
       is analyzed on its own merits.

    Returns a dictionary with the list of non-interactive indices and
    the corresponding reasons for each solution.
    """
    # --- Step 1: Analyze the starter code to establish context ---
    starter_code = example.get("starter_code")
    starter_is_interactive, starter_reason = analyze_interaction_strict(starter_code)

    sols: list[str] = example.get("solutions_list") or []

    # --- Step 2: If starter code is interactive, all solutions are interactive ---
    if starter_is_interactive is True:
        return {
            "non_interactive_idx": [-1],
            "interaction_reason": [f"starter_code:{starter_reason}"] * len(sols),
        }

    # --- Step 3: If context is not interactive, analyze solutions individually ---
    if not sols:
        return {"non_interactive_idx": [], "interaction_reason": []}

    flags, reasons = zip(*(analyze_interaction_strict(s) for s in sols))

    # A solution is non-interactive only if its flag is explicitly False.
    # `None` (for syntax errors) is not considered non-interactive.
    non_interactive_idx = [i for i, needs_tty in enumerate(flags) if needs_tty is False]

    # If the list is empty, it means no solution was provably non-interactive.
    if not non_interactive_idx and sols:
        non_interactive_idx = [-1]

    return {
        "non_interactive_idx": non_interactive_idx,
        "interaction_reason": list(reasons),
    }

In [22]:
new_sanitized_ds = new_sanitized_ds.map(
    detect_requires_input, batched=False, desc="static-analysis: stdin/tty detector"
)

In [23]:
new_sanitized_ds

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason'],
        num_rows: 8504
    })
    test: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason'],
        num_rows: 1496
    })
})

In [24]:
new_sanitized_ds = new_sanitized_ds.map(
    lambda ex: {"requires_input()": ex["non_interactive_idx"] == [-1]}, batched=False
)

In [25]:
# Filter dataset to only include examples where requires_input() is False
filtered_ds = new_sanitized_ds.filter(lambda x: not x["requires_input()"])
input_bearing_ds = new_sanitized_ds.filter(lambda x: x["requires_input()"])

In [26]:
filtered_ds

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason', 'requires_input()'],
        num_rows: 2973
    })
    test: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason', 'requires_input()'],
        num_rows: 331
    })
})

In [27]:
input_bearing_ds

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason', 'requires_input()'],
        num_rows: 5531
    })
    test: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason', 'requires_input()'],
        num_rows: 1165
    })
})

In [28]:
print(filtered_ds['train']['solutions_list'][7][0])

class Solution:
    def maxScoreSightseeingPair(self, A: List[int]) -> int:
        curmaxsight = A[0] - 1
        curmaxpair = 0
        for sight in A[1:]:
            if sight + curmaxsight > curmaxpair:
                curmaxpair = sight + curmaxsight
            if sight > curmaxsight:
                curmaxsight = sight
            curmaxsight -= 1
        return curmaxpair
            



In [29]:
print(filtered_ds['train']['starter_code'][0])


class Solution:
    def maxScore(self, cardPoints: List[int], k: int) -> int:
        


In [30]:
filtered_ds['train'][7]['input_output']

'{"fn_name": "maxScoreSightseeingPair", "inputs": [[[8, 1, 5, 2, 6]]], "outputs": [11]}'

In [31]:
Counter(filtered_ds['train']['difficulty'])

Counter({'introductory': 2311, 'interview': 661, 'competition': 1})

In [43]:
# Find the sole 'competition' level problem in filtered_ds
competition_level_problem = filtered_ds['train'].filter(lambda x: x['difficulty'] == 'competition')
interview_level_problems = filtered_ds['train'].filter(lambda x: x['difficulty'] == 'interview')

Filter:   0%|          | 0/2973 [00:00<?, ? examples/s]

In [44]:
interview_level_problems

Dataset({
    features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason', 'requires_input()'],
    num_rows: 661
})

In [60]:
I= 4

In [61]:
print(interview_level_problems[I]['solutions_list'][0])

class Solution:
    def maxFreq(self, s: str, maxLetters: int, minSize: int, maxSize: int) -> int:
        n = len(s)
        count = collections.Counter(s[i : i + minSize] for i in range(0, n - minSize +  1))
        res = 0 
        for k, v in count.items():
            if len(set(k)) <= maxLetters:
                res = max(res, v)
        return res


In [62]:
print(interview_level_problems[I]['input_output'])

{"fn_name": "maxFreq", "inputs": [["\"aababcaab\"", 2, 3, 4]], "outputs": [2]}


In [63]:
print(interview_level_problems[I]['question'])

Given a string s, return the maximum number of ocurrences of any substring under the following rules:

The number of unique characters in the substring must be less than or equal to maxLetters.
The substring size must be between minSize and maxSize inclusive.

 
Example 1:
Input: s = "aababcaab", maxLetters = 2, minSize = 3, maxSize = 4
Output: 2
Explanation: Substring "aab" has 2 ocurrences in the original string.
It satisfies the conditions, 2 unique letters and size 3 (between minSize and maxSize).

Example 2:
Input: s = "aaaa", maxLetters = 1, minSize = 3, maxSize = 3
Output: 2
Explanation: Substring "aaa" occur 2 times in the string. It can overlap.

Example 3:
Input: s = "aabcabcab", maxLetters = 2, minSize = 2, maxSize = 3
Output: 3

Example 4:
Input: s = "abcde", maxLetters = 2, minSize = 3, maxSize = 3
Output: 0

 
Constraints:

1 <= s.length <= 10^5
1 <= maxLetters <= 26
1 <= minSize <= maxSize <= min(26, s.length)
s only contains lowercase English letters.


In [71]:
print(new_sanitized_ds['train'][I]['question'])

The sequence of $m$ integers is called the permutation if it contains all integers from $1$ to $m$ exactly once. The number $m$ is called the length of the permutation.

Dreamoon has two permutations $p_1$ and $p_2$ of non-zero lengths $l_1$ and $l_2$.

Now Dreamoon concatenates these two permutations into another sequence $a$ of length $l_1 + l_2$. First $l_1$ elements of $a$ is the permutation $p_1$ and next $l_2$ elements of $a$ is the permutation $p_2$. 

You are given the sequence $a$, and you need to find two permutations $p_1$ and $p_2$. If there are several possible ways to restore them, you should find all of them. (Note that it is also possible that there will be no ways.)


-----Input-----

The first line contains an integer $t$ ($1 \le t \le 10\,000$) denoting the number of test cases in the input.

Each test case contains two lines. The first line contains one integer $n$ ($2 \leq n \leq 200\,000$): the length of $a$. The second line contains $n$ integers $a_1, a_2, \ldots

In [72]:
print(new_sanitized_ds['train'][I]['solutions_list'][0])

def possible(a):
    ans = set()
    s = set()
    lmax = 0
    for i in range(len(a)):
        lmax = max(lmax, a[i])
        s.add(a[i])
        if lmax == i + 1 and len(s) == i + 1:
            ans.add(i + 1)
    return ans


t = int(input())
for case_num in range(t):
    n = int(input())
    a = list(map(int, input().split(' ')))
    left = possible(a)
    a.reverse()
    right = possible(a)
    ans = []
    for l in left:
        if n - l in right:
            ans.append(l)
    print(len(ans))
    for l in ans:
        print(l, n - l)



In [74]:
import json

json.loads(new_sanitized_ds['train'][I]['input_output'])

{'inputs': ['6\n5\n1 4 3 2 1\n6\n2 4 1 3 2 1\n4\n2 1 1 3\n4\n1 3 3 1\n12\n2 1 3 4 5 6 7 8 9 1 10 2\n3\n1 1 1\n'],
 'outputs': ['2\n1 4\n4 1\n1\n4 2\n0\n0\n1\n2 10\n0\n']}

In [76]:
import json

def try_json(str):
    try:
        return json.loads(str)
    except:
        return str

In [77]:
io = [try_json(new_sanitized_ds['train'][I]['input_output']) for I in range(len(new_sanitized_ds['train']))]

In [78]:
io

[{'inputs': ['3\n2 2 3\n4 3 7\n10 1 9\n'], 'outputs': ['1\n6\n-1\n']},
 {'inputs': ['5\n3\n1 1 1\n2 2 2\n3 3 3\n4\n1 2 1 2\n2 1 2 1\n3 4 3 4\n7\n1 3 3 1 1 1 1\n2 4 4 3 2 2 4\n4 2 2 2 4 4 2\n3\n1 2 1\n2 3 3\n3 1 2\n10\n1 1 1 2 2 2 3 3 3 1\n2 2 2 3 3 3 1 1 1 2\n3 3 3 1 1 1 2 2 2 3\n'],
  'outputs': ['1 2 3\n1 2 1 2\n1 3 4 1 2 1 4\n1 2 3\n1 2 1 2 3 2 3 1 3 2\n']},
 {'inputs': ['2\n4 1\n5 5 5 5\n3 2\n0 0 0\n'], 'outputs': ['10\n0\n']},
 {'inputs': ['3\n6\n4 5 1 3 2 6\n5\n5 3 1 2 4\n4\n1 4 3 2\n'],
  'outputs': ['101011\n11111\n1001\n']},
 {'inputs': ['6\n5\n1 4 3 2 1\n6\n2 4 1 3 2 1\n4\n2 1 1 3\n4\n1 3 3 1\n12\n2 1 3 4 5 6 7 8 9 1 10 2\n3\n1 1 1\n'],
  'outputs': ['2\n1 4\n4 1\n1\n4 2\n0\n0\n1\n2 10\n0\n']},
 {'inputs': ['2\n4 6\n1 2\n1 3\n2 3\n2 4\n3 4\n3 4\n7 6\n1 2\n1 3\n2 4\n2 5\n3 6\n3 7\n'],
  'outputs': ['2\n3 4 \n4\n4 5 6 7 \n']},
 {'inputs': ['3\n3\n1 5\n2 10\n2 8\n7\n0 1\n3 1\n1 1\n6 1\n1 1\n4 1\n4 1\n6\n2 6\n2 3\n2 8\n2 7\n4 4\n5 5\n'],
  'outputs': ['8\n0\n7\n']},
 {'inputs': [

In [83]:
print(interview_level_problems[0]['question'])

There are several cards arranged in a row, and each card has an associated number of points The points are given in the integer array cardPoints.
In one step, you can take one card from the beginning or from the end of the row. You have to take exactly k cards.
Your score is the sum of the points of the cards you have taken.
Given the integer array cardPoints and the integer k, return the maximum score you can obtain.
 
Example 1:
Input: cardPoints = [1,2,3,4,5,6,1], k = 3
Output: 12
Explanation: After the first step, your score will always be 1. However, choosing the rightmost card first will maximize your total score. The optimal strategy is to take the three cards on the right, giving a final score of 1 + 6 + 5 = 12.

Example 2:
Input: cardPoints = [2,2,2], k = 2
Output: 4
Explanation: Regardless of which two cards you take, your score will always be 4.

Example 3:
Input: cardPoints = [9,7,7,9,7,7,9], k = 7
Output: 55
Explanation: You have to take all the cards. Your score is the su

In [81]:
print(interview_level_problems[0]['solutions_list'][0])

class Solution:
    def maxScore(self, cardPoints: List[int], k: int) -> int:
        max_score = 0
        curr_score= 0
        init_hand = cardPoints[len(cardPoints)-k:]
        max_score = sum(init_hand)
        curr_score = max_score
        for i in range(k):
            curr_score -= init_hand[i]
            curr_score += cardPoints[i]
            if curr_score > max_score:
                max_score = curr_score
        return max_score


In [82]:
print(interview_level_problems[0]['input_output'])

{"fn_name": "maxScore", "inputs": [[[1, 2, 3, 4, 5, 6, 1], 3]], "outputs": [12]}


In [84]:
new_sanitized_ds.filter(lambda x: x['input_output'] == '')

Filter:   0%|          | 0/8504 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1496 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason', 'requires_input()'],
        num_rows: 164
    })
    test: Dataset({
        features: ['problem_id', 'question', 'solutions', 'input_output', 'difficulty', 'url', 'starter_code', 'solutions_list', 'n_sols', 'non_interactive_idx', 'interaction_reason', 'requires_input()'],
        num_rows: 31
    })
})

In [ ]:
new_sanitized_ds.push_to_hub("jvelja/apps_sanitized")

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"attn_implementation": "eager", "device_map": "auto", "torch_dtype": torch.float16},
    tokenizer_kwargs={"padding_side": "left"},
)

questions = new_sanitized_ds['train']['question']

In [ ]:
model.compile(mode="reduce-overhead")

In [ ]:
with torch.inference_mode():                        # no gradients
    emb = model.encode(questions, batch_size=1,
                       prompt_name="query",         # keeps instruction prompt
                       normalize_embeddings=True,
                       convert_to_tensor=False)

In [ ]:
from collections import Counter


def bin_length(length: int) -> str:
    """Bin the length of non_interactive_idx into specified ranges."""
    if 1 <= length < 2:
        return "1"
    elif 2 <= length < 5:
        return "2-4"
    elif 5 <= length < 10:
        return "5-9"
    else:
        return "10+"


lengths = [len(filtered_ds["train"][i]["non_interactive_idx"]) for i in range(len(filtered_ds["train"]))]
binned = [bin_length(l) for l in lengths]
Counter(binned)

In [ ]:
input_bearing_lengths = [
    len(input_bearing_ds["train"][i]["solutions_list"]) for i in range(len(input_bearing_ds["train"]))
]
input_bearing_binned = [bin_length(l) for l in input_bearing_lengths]
Counter(input_bearing_binned)

In [ ]:
def estimate_prices(
    num_tokens_input: int,
    num_tokens_output: int,
    number_of_prompts: int = 1,
    price_per_token_input: float = 1.1 / 1_000_000,
    price_per_token_output: float = 4.4 / 1_000_000,
) -> float:
    """Estimate the cost of running a model for a given number of tokens."""
    return number_of_prompts * (num_tokens_input * price_per_token_input + num_tokens_output * price_per_token_output)

In [ ]:
estimate_prices(
    num_tokens_input=2500,
    num_tokens_output=3500,
    number_of_prompts=10_000,
)

In [ ]:
estimate_prices(
    num_tokens_input=2500,
    num_tokens_output=3500,
    number_of_prompts=10_000,
    price_per_token_input=0.13 / 1_000_000,
    price_per_token_output=0.8 / 1_000_000,
)

In [ ]:
estimate_prices(
    num_tokens_input=2500,
    num_tokens_output=3500,
    number_of_prompts=10_000,
    price_per_token_input=0.4 / 1_000_000,
    price_per_token_output=1.6 / 1_000_000,
)

# Prices

- GPT o4-mini: ~180USD for processing the entire dataset (fix problem specification + starter code)
- qwen/qwen3-235b-a22b-2507: ~40 USD for processing the entire dataset (fix problem specification + starter code)
- GPT 4.1-mini: ~50 USD for processing the entire dataset (fix problem specification + starter code)

In [ ]:
filtered_ds.push_to_hub("jvelja/apps_reshuffled_filtered")

In [ ]:
input_bearing_ds.push_to_hub("jvelja/apps_reshuffled_tobe_transformed")

In [ ]:
input_bearing_ds["train"][0]

In [ ]:
I = 2972
print(f"{I}th example - Starter code:")
print(filtered_ds["train"][I]["starter_code"] if filtered_ds["train"][I]["starter_code"] != "" else "No starter code")
print("--------------------------------")
print(f"{I}th example - Non-interactive solution list indices:")
print(filtered_ds["train"][I]["non_interactive_idx"])
idx = filtered_ds["train"][I]["non_interactive_idx"][0]
print("--------------------------------")
print(f"{I}th example - Solution number {idx + 1}:")
print(filtered_ds["train"][I]["solutions_list"][idx])
print("--------------------------------")
print(f"{I}th example - Interaction reason:")
print(filtered_ds["train"][I]["interaction_reason"])

In [ ]:
filtered_ds["train"][I]["solutions_list"]

In [ ]:
# Get empty-solution datapoints in filtered_ds
empty_solution_ds = filtered_ds.filter(lambda x: x["solutions_list"] == [""])

In [ ]:
filtered_ds = filtered_ds.filter(lambda x: x["solutions_list"] != [""])

In [ ]:
filtered_ds

In [ ]:
# For all filtered_ds entries, retain up to 5 solutions (sampled at random), conditional on the interaction reason being None

In [ ]:
I = 2900
print(f"{I}th example - Question:")
print(filtered_ds["train"][I]["question"])
print("--------------------------------")
print(f"{I}th example - Starter code:")
print(filtered_ds["train"][I]["starter_code"] if filtered_ds["train"][I]["starter_code"] != "" else "No starter code")
print("--------------------------------")
print(f"{I}th example - Non-interactive solution list indices:")
print(filtered_ds["train"][I]["non_interactive_idx"])
idx = filtered_ds["train"][I]["non_interactive_idx"][0]
print("--------------------------------")
for idx in filtered_ds["train"][I]["non_interactive_idx"]:
    print(f"{I}th example - Solution number {idx + 1}:")
    print(filtered_ds["train"][I]["solutions_list"][idx])
    print("-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-")
print("--------------------------------")


# print(f"{I}th example - Interaction reason:")
# print(filtered_ds["train"][I]["interaction_reason"])

In [ ]:
unique_problems = len(filtered_ds["train"])
solutions_count = sum(len(row["solutions_list"]) for row in filtered_ds["train"])
print(f"Unique problems: {unique_problems}")
print(f"Solutions count: {solutions_count}")

In [ ]:
# Get number of solutions that are not associated with an interaction reason (i.e., their interaction_reason is None)
sum(
    [
        sum(
            [
                filtered_ds["train"][i]["interaction_reason"][h] is None
                for h in range(len(filtered_ds["train"][i]["interaction_reason"]))
            ]
        )
        for i in range(len(filtered_ds["train"]))
    ]
)

In [ ]:
empty_solution_ds

In [ ]:
interaction_finder = InteractionFinder()
tree = ast.parse(filtered_ds["train"][I]["solutions_list"][1])
interaction_finder.visit(tree)

In [ ]:
interaction_finder.found

In [ ]:
filtered_ds["train"][]

In [ ]:
empty_soln_ds = filtered_ds.filter(lambda x: x["solutions_list"] == [""])

In [ ]:
empty_soln_ds

In [ ]:
print(filtered_ds["train"]["solutions_list"][1][21])

In [ ]:
new_sanitized_ds.push_to_hub("jvelja/apps_reshuffled")

In [ ]:
filtered_ds.push_to_hub("jvelja/apps_reshuffled_filtered")

In [ ]:
filtered_ds

In [ ]:
Counter(new_sanitized_ds["train"]["requires_input()"])

In [ ]:
print(new_sanitized_ds["train"]["solutions_list"][0][6])

In [ ]:
print(new_sanitized_ds["train"]["solutions_list"][0][1])